# EWC Space Delta-V · Technical Deep-Dive · A small DSMC solver

One-dimensional direct simulation Monte Carlo (DSMC) between two plates, checked against the continuum and free-molecular limits.

**Code status: working demo.** Python 3.9 or later with NumPy (already installed in Colab). No other dependency.

**What it does.** It simulates an argon-like hard-sphere gas between two parallel, diffusely reflecting plates at 310 K and 290 K, in one space dimension and three velocity dimensions, across the whole range of Knudsen numbers. It compares the heat flux carried by the particles with the two analytic limits (continuum Fourier conduction and the free-molecular limit), runs a statistical-noise study, prints the cost scaling of DSMC and then runs a self-test.

**Run length.** The run cell near the end has a `QUICK` switch. Quick mode takes a few minutes; set `QUICK = False` for the full run (about 11 minutes on the machine used for the article; Colab may differ).

**Precaution.** This is an educational model with stated assumptions, not a mission-grade tool and not any company's solver. Test it in your own environment and have qualified experts review it before any implementation. It is provided without warranty of fitness for any real mission, safety-critical or operational use.

Assumed values are marked `ASSUMED` in the code. The collision-pair selection is a vectorised "disjoint pairs" variant of Bird's no-time-counter (NTC) scheme: each particle takes part in at most one collision per time step.

## 1. Imports
Standard library modules plus NumPy, which does the vectorised particle arithmetic.

In [ ]:
import argparse
import json
import math
import sys
import time

import numpy as np

## 2. Constants
Boltzmann constant, argon atom mass and the hard-sphere diameter. The diameter is chosen to reproduce argon's viscosity at 273 K (an assumption), and it fixes the collision cross-section.

In [ ]:
K_B = 1.380649e-23          # Boltzmann constant, J/K
M = 6.63e-26                # argon atom mass, kg
D = 3.64e-10                # hard-sphere diameter, m: ASSUMED, chosen to reproduce argon's viscosity (2.1e-5 Pa s at 273 K)
SIGMA = math.pi * D ** 2    # total collision cross-section, m^2

## 3. Analytic ingredients
The exact results the simulation is judged against: mean free path, mean speed, hard-sphere viscosity and conductivity, collision frequency, the continuum and free-molecular heat fluxes, the series-resistance bridge between them, the cost scaling of DSMC with Knudsen number, and the radius at which a point-source plume reaches Kn = 1.

In [ ]:
# ------------------------------------------------------------------ analytic ingredients
def mean_free_path(n):
    """Hard-sphere mean free path, lambda = 1 / (sqrt(2) pi d^2 n)."""
    return 1.0 / (math.sqrt(2.0) * SIGMA * n)


def mean_speed(t):
    return math.sqrt(8.0 * K_B * t / (math.pi * M))


def viscosity_hs(t):
    return (5.0 / 16.0) * math.sqrt(math.pi * M * K_B * t) / (math.pi * D ** 2)


def conductivity_hs(t):
    """Chapman-Enskog hard-sphere thermal conductivity: (15/4)(k_B/m) mu = (75/64)(k_B/d^2) sqrt(k_B T/(pi m))."""
    return 3.75 * (K_B / M) * viscosity_hs(t)


def collision_frequency(n, t):
    """Mean collisions per particle per second, sqrt(2) n sigma c_bar."""
    return math.sqrt(2.0) * n * SIGMA * mean_speed(t)


def q_continuum(t1, t2, length):
    return conductivity_hs(0.5 * (t1 + t2)) * (t1 - t2) / length


def q_free_molecular(n0, t1, t2):
    """Free-molecular heat flux, diffuse full-accommodation walls, zero net particle flux."""
    gamma = 2.0 * n0 * math.sqrt(K_B / (2.0 * math.pi * M)) / (t1 ** -0.5 + t2 ** -0.5)
    return 2.0 * K_B * gamma * (t1 - t2)


def q_bridge(q_c, q_fm):
    """Heuristic series-resistance bridge between the two limits (a comparison curve, not a theory)."""
    return 1.0 / (1.0 / q_c + 1.0 / q_fm)


def dsmc_cost(kn, dims=3, length=1.0, ppc=30):
    """Cost of resolving a flow at Knudsen number kn: cells of lambda/3, ppc particles per cell (at least 30 is the usual guidance), steps of 0.2 lambda/c_bar."""
    cells_per_dim = math.ceil(3.0 / kn)
    cells = cells_per_dim ** dims
    particles = ppc * cells
    steps = 5.0 / kn                       # steps to advance one flow-through time L / c_bar
    return cells, particles, steps, particles * steps


def breakdown_radius(n_exit, r_exit, d=3.7e-10):
    """Point-source plume, n(r) = n_e (r_e / r)^2: the distance at which Kn = lambda / r reaches 1.
    Kn(r) = r / (sqrt(2) sigma n_e r_e^2).  The inputs are ASSUMED, not any thruster's data; real plumes are not point sources."""
    sig = math.pi * d ** 2
    return math.sqrt(2.0) * sig * n_exit * r_exit ** 2

## 4. The DSMC solver
`run_plates` is the solver. Each time step moves the particles (re-emitting them from the wall temperature at a wall), sorts them into cells, selects and accepts candidate collision pairs in each cell, scatters the accepted pairs isotropically (conserving momentum and energy) and, once the flow is steady, samples the heat flux and the temperature profile.

In [ ]:
# ------------------------------------------------------------------ the DSMC solver
def run_plates(kn, n_part, t1=310.0, t2=290.0, length=0.01, relax=None, sample=None, seed=1, step_frac=0.2):
    """1-D DSMC between a hot plate (x = 0, T1) and a cold plate (x = L, T2). Returns a dict of results."""
    rng = np.random.default_rng(seed)
    tm = 0.5 * (t1 + t2)
    lam = kn * length
    n0 = 1.0 / (math.sqrt(2.0) * SIGMA * lam)          # mean number density that gives this Knudsen number
    fnum = n0 * length / n_part                       # real molecules per simulated particle, per unit area
    cbar = mean_speed(tm)
    dt = min(step_frac * lam / cbar, 0.2 * length / cbar)
    nc = int(min(max(math.ceil(3.0 * length / lam), 20), 100))      # cells of about lambda / 3 (20 to 100 cells)
    dx = length / nc
    if relax is None:                                  # about 4 conduction times plus a few crossing times, capped
        diffusivity = 1.5 * viscosity_hs(tm) / (n0 * M)
        relax = int(min(max(6.0 * length ** 2 / diffusivity / dt, 1500), 12000))
    if sample is None:
        sample = max(relax, 3000)

    # initial state: uniform pressure, linear temperature profile
    u = rng.random(n_part)
    x = length * (np.exp(u * math.log(t2 / t1)) - 1.0) * t1 / (t2 - t1)
    tx = t1 + (t2 - t1) * x / length
    v = rng.normal(size=(n_part, 3)) * np.sqrt(K_B * tx / M)[:, None]

    srmax = np.full(nc, SIGMA * 4.0 * cbar)
    rem = np.zeros(nc)
    e_in = np.zeros(2)
    e_out = np.zeros(2)
    prof_n = np.zeros(nc)
    prof_v = np.zeros((nc, 3))
    prof_v2 = np.zeros(nc)
    n_coll = 0

    for step in range(relax + sample):
        sampling = step >= relax
        # --- free flight, then diffuse re-emission at the walls (repeated if a particle crosses more than once)
        x = x + v[:, 0] * dt
        out = (x < 0.0) | (x > length)
        if out.any():
            idx = np.nonzero(out)[0]
            xo = x[idx]
            vo = v[idx].copy()
            while idx.size:
                left = xo < 0.0
                wall = np.where(left, 0.0, length)
                tw = np.where(left, t1, t2)
                tau = np.abs(xo - wall) / np.abs(vo[:, 0])         # time left after the wall hit
                ke_old = 0.5 * M * (vo ** 2).sum(axis=1)
                vn = np.sqrt(-2.0 * K_B * tw / M * np.log(1.0 - rng.random(idx.size)))   # flux-weighted normal speed
                vt = rng.normal(size=(idx.size, 2)) * np.sqrt(K_B * tw / M)[:, None]
                vnew = np.column_stack([np.where(left, vn, -vn), vt])
                ke_new = 0.5 * M * (vnew ** 2).sum(axis=1)
                if sampling:
                    for w, sel in ((0, left), (1, ~left)):
                        e_in[w] += ke_old[sel].sum()
                        e_out[w] += ke_new[sel].sum()
                xo = wall + vnew[:, 0] * tau
                v[idx] = vnew
                x[idx] = xo
                vo = vnew
                again = (xo < 0.0) | (xo > length)
                idx, xo, vo = idx[again], xo[again], vo[again]

        # --- collisions: index particles by cell, select candidate pairs (NTC), accept, scatter isotropically
        cell = np.minimum((x / dx).astype(np.int64), nc - 1)
        order = np.lexsort((rng.random(n_part), cell))              # random order inside each cell
        counts = np.bincount(cell, minlength=nc)
        off = np.cumsum(counts) - counts
        nsel = 0.5 * counts * np.maximum(counts - 1, 0) * fnum * srmax * dt / dx + rem
        ntr = np.minimum(np.floor(nsel).astype(np.int64), counts // 2)
        rem = np.minimum(nsel - ntr, 1.0)
        tot = int(ntr.sum())
        if tot:
            pc = np.repeat(np.arange(nc), ntr)
            k = np.arange(tot) - np.repeat(np.cumsum(ntr) - ntr, ntr)
            i1 = order[off[pc] + 2 * k]
            i2 = order[off[pc] + 2 * k + 1]
            cr = np.sqrt(((v[i1] - v[i2]) ** 2).sum(axis=1))
            sr = SIGMA * cr
            np.maximum.at(srmax, pc, sr)
            acc = rng.random(tot) < sr / srmax[pc]
            if acc.any():
                a1, a2, c = i1[acc], i2[acc], cr[acc]
                vcm = 0.5 * (v[a1] + v[a2])
                cth = 2.0 * rng.random(c.size) - 1.0
                sth = np.sqrt(1.0 - cth ** 2)
                ph = 2.0 * math.pi * rng.random(c.size)
                nvec = np.column_stack([cth, sth * np.cos(ph), sth * np.sin(ph)])
                v[a1] = vcm + 0.5 * c[:, None] * nvec
                v[a2] = vcm - 0.5 * c[:, None] * nvec
                n_coll += int(acc.sum())

        if sampling:
            prof_n += counts
            np.add.at(prof_v, cell, v)
            np.add.at(prof_v2, cell, (v ** 2).sum(axis=1))

    t_sample = sample * dt
    q_hot = (e_out[0] - e_in[0]) * fnum / t_sample          # energy per area per time from hot wall into gas
    q_cold = (e_in[1] - e_out[1]) * fnum / t_sample         # energy per area per time from gas into cold wall
    nn = np.maximum(prof_n, 1.0)
    mean_v = prof_v / nn[:, None]
    tprof = M / (3.0 * K_B) * (prof_v2 / nn - (mean_v ** 2).sum(axis=1))
    nu_meas = 2.0 * n_coll / (n_part * (relax + sample) * dt)
    return dict(kn=kn, q=0.5 * (q_hot + q_cold), q_hot=q_hot, q_cold=q_cold, nc=nc, dt=dt, n0=n0,
                steps=relax + sample, nu_meas=nu_meas, nu_theory=collision_frequency(n0, tm), tprof=tprof.tolist())

## 5. Noise study
Repeats a run with independent random seeds at several particle numbers, to measure how the scatter of the heat flux falls with particle number.

In [ ]:
def noise_study(kn, sizes, seeds, steps=(800, 1600)):
    """Standard deviation of the measured heat flux (relative to the mean) against particle number."""
    out = []
    for n in sizes:
        qs = [run_plates(kn, n, relax=steps[0], sample=steps[1], seed=100 + s)["q"] for s in range(seeds)]
        out.append(dict(n=n, mean=float(np.mean(qs)), rel_std=float(np.std(qs, ddof=1) / np.mean(qs))))
    return out

## 6. The study
`study` runs the simulation across Knudsen numbers and prints the six result tables: heat flux, temperature step at the walls, statistical noise, cost scaling, plume breakdown distance and a collision-frequency check.

In [ ]:
# ------------------------------------------------------------------ report
def study(quick=False):
    t1, t2, length = 310.0, 290.0, 0.01
    kns = [0.2, 1.0, 5.0] if quick else [0.05, 0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 20.0]
    n_part = 10000 if quick else 30000
    rows = []
    print("== A. Heat flux between plates (T1 = %.0f K, T2 = %.0f K, L = %.3f m, %d particles) ==" % (t1, t2, length, n_part))
    print("   Kn      cells  q_DSMC (W/m2)  q_cont (W/m2)  q_fm (W/m2)  q_bridge   DSMC/cont  DSMC/bridge  hot/cold flux")
    for kn in kns:
        r = run_plates(kn, n_part, t1, t2, length)
        qc = q_continuum(t1, t2, length)
        qf = q_free_molecular(r["n0"], t1, t2)
        qb = q_bridge(qc, qf)
        r.update(q_cont=qc, q_fm=qf, q_bridge=qb)
        rows.append(r)
        print(f"   {kn:5.2f}  {r['nc']:5d}  {r['q']:13.4e}  {qc:13.4e}  {qf:11.4e}  {qb:9.3e}  {r['q']/qc:9.3f}  {r['q']/qb:11.3f}  {r['q_hot']/r['q_cold']:.3f}")

    mid = min(rows, key=lambda r: abs(r["kn"] - 0.1)) if not quick else rows[0]
    tp = mid["tprof"]
    jump_hot, jump_cold = t1 - tp[0], tp[-1] - t2
    print(f"\n== B. Temperature profile at Kn = {mid['kn']}: hot wall {t1:.0f} K, first cell {tp[0]:.1f} K, last cell {tp[-1]:.1f} K, cold wall {t2:.0f} K ==")
    print(f"   Gas-wall temperature step in the end cells: {jump_hot:.2f} K (hot) and {jump_cold:.2f} K (cold); mean {0.5*(jump_hot+jump_cold):.2f} K out of {t1 - t2:.0f} K")

    print("\n== C. Statistical noise at Kn = 1: relative standard deviation of the heat flux against particle number ==")
    sizes = [1000, 4000, 16000] if quick else [2000, 8000, 32000]
    noise = noise_study(1.0, sizes, 5 if quick else 8)
    for r in noise:
        print(f"   N = {r['n']:6d}: relative std = {r['rel_std']*100:6.3f} %   (std x sqrt(N) = {r['rel_std']*math.sqrt(r['n']):.3f})")
    slope = float(np.polyfit(np.log([r["n"] for r in noise]), np.log([r["rel_std"] for r in noise]), 1)[0])
    print(f"   Fitted exponent of std against N: {slope:.2f}  (1/sqrt(N) scaling predicts -0.50)")

    print("\n== D. DSMC cost against Knudsen number (3-D, L = 1 m, cells of lambda/3, 30 particles per cell, steps of 0.2 lambda/c) ==")
    print("   Kn        cells         particles        steps per L/c    particle-steps    relative to Kn = 0.1")
    base = dsmc_cost(0.1)[3]
    cost = []
    for kn in (1.0, 0.1, 0.01, 0.001):
        c, p, s, w = dsmc_cost(kn)
        cost.append(dict(kn=kn, cells=c, particles=p, steps=s, work=w, rel=w / base))
        print(f"   {kn:6.3f}  {c:12.3e}  {p:14.3e}  {s:14.1f}  {w:15.3e}  {w/base:12.3e}")
    print(f"   Work ratio per decade of Kn (3-D): {dsmc_cost(0.01)[3]/dsmc_cost(0.1)[3]:.0f} x (scaling Kn^-4)")

    print("\n== E. Where a point-source plume reaches Kn = 1 (exit density and exit radius ASSUMED, d = 3.7 A) ==")
    print("   Kn = 1 is reached at the distance r (m) below; the answer scales linearly with the exit density and with the square of the exit radius")
    print("   exit density (1/m3)   r_exit 0.02 m   r_exit 0.05 m   r_exit 0.10 m")
    plume = []
    for n_e in (1e21, 1e22, 1e23):
        vals = [breakdown_radius(n_e, r_e) for r_e in (0.02, 0.05, 0.10)]
        plume.append(dict(n_exit=n_e, r_exit=[0.02, 0.05, 0.10], r_break=vals))
        print(f"   {n_e:10.0e}        {vals[0]:12.2f}   {vals[1]:12.2f}   {vals[2]:12.2f}")

    print("\n== F. Collision-frequency check ==")
    chk = run_plates(0.5, 8000, t1, t2, length)
    print(f"   Kn = 0.5: measured {chk['nu_meas']:.4e} per particle per second, kinetic theory {chk['nu_theory']:.4e} ({chk['nu_meas']/chk['nu_theory']*100:.1f} %)")
    return dict(rows=rows, noise=noise, noise_slope=slope, jump_hot=jump_hot, jump_cold=jump_cold, cost=cost, plume=plume, nu_check=dict(meas=chk["nu_meas"], theory=chk["nu_theory"]),
                t1=t1, t2=t2, length=length, n_part=n_part)

## 7. The self-test
Twelve checks against handbook values and the analytic limits, including argon's viscosity and conductivity, the free-molecular limit at high Knudsen number, kinetic-theory collision frequency and energy balance.

In [ ]:
def selftest():
    count = [0]

    def check(cond, msg):
        count[0] += 1
        assert cond, msg

    # textbook transport coefficients of argon at 273 K: viscosity ~2.1e-5 Pa s, conductivity ~0.0164 W/m/K
    check(abs(viscosity_hs(273.0) / 2.1e-5 - 1.0) < 0.03, "viscosity")
    check(abs(conductivity_hs(273.0) / 0.0164 - 1.0) < 0.04, "conductivity")
    # mean free path of argon at 1 atm, 273 K is about 6e-8 m
    n_atm = 101325.0 / (K_B * 273.0)
    check(5e-8 < mean_free_path(n_atm) < 8e-8, "mean free path")
    # cost scaling: three-dimensional work grows by about 10^4 per decade of Knudsen number
    check(0.8e4 < dsmc_cost(0.01)[3] / dsmc_cost(0.1)[3] < 1.3e4, "cost scaling")
    # the bridge formula reduces to the smaller limit
    check(abs(q_bridge(1.0, 1e9) - 1.0) < 1e-6 and abs(q_bridge(1e9, 2.0) - 2.0) < 1e-6, "bridge limits")
    # free-molecular limit: DSMC at Kn = 30 agrees within 5 %, and the wall fluxes balance within 3 %
    r = run_plates(30.0, 6000, seed=7)
    qf = q_free_molecular(r["n0"], 310.0, 290.0)
    check(abs(r["q"] / qf - 1.0) < 0.05, "free-molecular limit")
    check(abs(r["q_hot"] / r["q_cold"] - 1.0) < 0.03, "energy balance at Kn = 30")
    # kinetic-theory collision frequency reproduced within 4 % in the transitional regime
    r = run_plates(0.5, 12000, seed=7)
    check(abs(r["nu_meas"] / r["nu_theory"] - 1.0) < 0.04, "collision frequency")
    # transitional regime: the continuum prediction must overestimate strongly, and the bridge must be close
    qc = q_continuum(310.0, 290.0, 0.01)
    check(r["q"] < 0.6 * qc, "continuum overestimates at Kn = 0.5")
    check(abs(r["q"] / q_bridge(qc, q_free_molecular(r["n0"], 310.0, 290.0)) - 1.0) < 0.12, "bridge within 12 %")
    check(abs(r["q_hot"] / r["q_cold"] - 1.0) < 0.03, "energy balance at Kn = 0.5")
    # point-source breakdown radius grows with the square of the exit radius
    check(abs(breakdown_radius(1e22, 0.1) / breakdown_radius(1e22, 0.05) - 4.0) < 1e-9, "breakdown radius scaling")
    print(f"\nSELF-TEST PASSED ({count[0]} checks)")

## 8. Run it
Choose the run length, then run the study.

In [ ]:
QUICK = True   # set to False for the full run (about 11 minutes)

In [ ]:
results = study(quick=QUICK)

## 9. Run the self-test

In [ ]:
selftest()